# VDMS functional walkthrough

This notebook is a self-contained, manual walkthrough of the **VDMS** functional-test flow.
It mirrors the automated logic in:

- `tests/functional/conftest.py`
- `tests/functional/data.py`
- `tests/functional/filter_assertions.py`
- `tests/functional/test_vdms_filters.py`

## Prerequisites

- Run from this repository checkout in an environment with `docker` and Docker Compose available.
- The Docker daemon must be running and able to build/pull the images referenced by the compose files.
- The notebook talks to the same compose stack used by the functional tests: `docker/compose.yaml` plus `docker/compose.vdms.yaml`.

## What this notebook demonstrates

1. Load deterministic fixture documents and filter cases from the functional-test source of truth.
2. Define the VDMS-specific ports and environment variables used by the tests.
3. Start the compose stack, wait for readiness, seed fixture data, and **restart `vector-retriever` after seeding** to match the VDMS test quirk.
4. Verify the seed is queryable.
5. Exercise `/health`, `/ready`, `/capabilities/filters`, and `/query` with the same logical cases covered by the automated suite.
6. Tear the stack down cleanly when exploration is complete.


In [8]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from copy import deepcopy
from pathlib import Path
from typing import Any
from uuid import uuid4

import requests

REPO_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repository root from the notebook working directory.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tests.functional.data import DOCUMENTS, FILTER_CASES

FILTER_CASE_LOOKUP = {case["name"]: case for case in FILTER_CASES}

print(f"Repo root: {REPO_ROOT}")
print(f"Loaded {len(DOCUMENTS)} deterministic documents and {len(FILTER_CASES)} filter cases.")


Repo root: /home/intel/workbench/integration/lib/microservices/vector-retriever/vector-retriever
Loaded 6 deterministic documents and 20 filter cases.


## Fixture dataset preview

The functional tests use six deterministic documents with intentionally varied metadata fields.
This quick preview makes it easy to see which IDs and attributes are expected to match later filter calls.


In [9]:
fixture_preview = [
    {
        "video_id": doc["metadata"]["video_id"],
        "camera_zone": doc["metadata"]["camera_zone"],
        "description": doc["metadata"]["description"],
        "prefix": doc["metadata"]["prefix"],
        "frame_number": doc["metadata"]["frame_number"],
        "bucket_name": doc["metadata"]["bucket_name"],
        "tags": doc["metadata"]["tags"],
        "created_at": doc["metadata"]["created_at"],
        "has_optional_note": "optional_note" in doc["metadata"],
    }
    for doc in DOCUMENTS
]
print(json.dumps(fixture_preview, indent=2))


[
  {
    "video_id": "vid-001",
    "camera_zone": "north",
    "description": "red car at intersection",
    "prefix": "alpha-route",
    "frame_number": 10,
    "bucket_name": "cam-a",
    "tags": [
      "traffic",
      "vehicle",
      "red"
    ],
    "created_at": "2026-03-10T10:00:00+00:00",
    "has_optional_note": true
  },
  {
    "video_id": "vid-002",
    "camera_zone": "south",
    "description": "blue bus near station",
    "prefix": "beta-route",
    "frame_number": 20,
    "bucket_name": "cam-b",
    "tags": [
      "traffic",
      "vehicle",
      "bus"
    ],
    "created_at": "2026-03-15T12:00:00+00:00",
    "has_optional_note": false
  },
  {
    "video_id": "vid-003",
    "camera_zone": "east",
    "description": "person with bicycle crossing",
    "prefix": "alpha-bike",
    "frame_number": 30,
    "bucket_name": "cam-c",
    "tags": [
      "pedestrian",
      "bicycle"
    ],
    "created_at": "2026-03-20T08:30:00+00:00",
    "has_optional_note": true
  },
  

## VDMS-specific environment and port settings

These values mirror the VDMS branch of `PORT_MAP` and the environment-building logic in `tests/functional/conftest.py`.
The unique index name keeps each notebook run isolated from other local sessions.


In [10]:
BACKEND = "vdms"
QUERY_TEXT = "traffic light malfunctio"
COMPOSE_FILES = ["docker/compose.yaml", f"docker/compose.{BACKEND}.yaml"]
COMPOSE_PROJECT = "docker"
OV_MODELS_SOURCE_VOLUME = "docker_ov_models"
OV_MODELS_DEST_VOLUME = f"{COMPOSE_PROJECT}_ov-models"
DEFAULT_TEST_EMBEDDING_MODEL_NAME = "CLIP/clip-vit-b-32"

PORT_SETTINGS = {
    "VECTOR_RETRIEVER_HOST_PORT": "6101",
    "EMBEDDING_SERVER_PORT": "9711",
    "VDMS_VDB_HOST_PORT": "5511",
}

NOTEBOOK_STATE = {
    "backend": BACKEND,
    "env": None,
    "base_url": None,
}

print(json.dumps(PORT_SETTINGS, indent=2))


{
  "VECTOR_RETRIEVER_HOST_PORT": "6101",
  "EMBEDDING_SERVER_PORT": "9711",
  "VDMS_VDB_HOST_PORT": "5511"
}


## Local helper functions

All notebook helper logic lives in this cell so the walkthrough stays self-contained.
The helpers intentionally mirror the functional-test behavior for compose execution, readiness polling,
deterministic seeding, restart-after-seed handling for VDMS, API invocation, and human-friendly result display.


In [11]:
def build_env() -> dict[str, str]:
    env = os.environ.copy()
    env.update(PORT_SETTINGS)
    env["RETRIEVER_BACKEND"] = BACKEND
    index_name = f"vr_functional_{BACKEND}_{uuid4().hex[:8]}"
    env["INDEX_NAME"] = index_name
    env["VS_INDEX_NAME"] = index_name
    env["EMBEDDING_MODEL_NAME"] = (
        env.get("EMBEDDING_MODEL_NAME", "").strip() or DEFAULT_TEST_EMBEDDING_MODEL_NAME
    )
    env["EMBEDDING_DEVICE"] = "CPU"
    env["EMBEDDING_USE_OV"] = "true"
    env["VECTOR_RETRIEVER_LOG_LEVEL"] = "debug"
    return env


def compose_command(args: list[str]) -> list[str]:
    command = ["docker", "compose"]
    for compose_file in COMPOSE_FILES:
        command.extend(["-f", compose_file])
    command.extend(args)
    return command


def run_compose(
    args: list[str],
    env: dict[str, str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    command = compose_command(args)
    print("$", " ".join(command))
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip(), file=sys.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(
            f"docker compose command failed (exit {completed.returncode})\n"
            f"cmd: {' '.join(command)}\n"
            f"stdout: {completed.stdout}\n"
            f"stderr: {completed.stderr}"
        )
    return completed


def ensure_ov_models_volume() -> None:
    if subprocess.run(
        ["docker", "volume", "inspect", OV_MODELS_DEST_VOLUME],
        capture_output=True,
        text=True,
    ).returncode == 0:
        print(f"OV models volume already present: {OV_MODELS_DEST_VOLUME}")
        return

    if subprocess.run(
        ["docker", "volume", "inspect", OV_MODELS_SOURCE_VOLUME],
        capture_output=True,
        text=True,
    ).returncode != 0:
        print(
            "Cached OV models source volume not found; the embedding service may load models at runtime."
        )
        return

    subprocess.run(
        ["docker", "volume", "create", OV_MODELS_DEST_VOLUME],
        check=True,
        capture_output=True,
        text=True,
    )
    subprocess.run(
        [
            "docker",
            "run",
            "--rm",
            "-v",
            f"{OV_MODELS_SOURCE_VOLUME}:/src",
            "-v",
            f"{OV_MODELS_DEST_VOLUME}:/dst",
            "alpine",
            "sh",
            "-c",
            "cp -r /src/. /dst/",
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    print(f"Seeded OV models volume: {OV_MODELS_DEST_VOLUME}")


def wait_for_ready(base_url: str, timeout_seconds: int = 600) -> dict[str, Any]:
    start = time.time()
    last_error = ""
    while time.time() - start < timeout_seconds:
        try:
            response = requests.get(f"{base_url}/ready", timeout=10)
            if response.status_code == 200:
                payload = response.json()
                if payload.get("status") == "ready":
                    print(f"Service ready at {base_url}")
                    return payload
                last_error = f"unexpected /ready payload: {payload}"
            else:
                last_error = f"/ready status={response.status_code} body={response.text}"
        except Exception as exc:  # noqa: BLE001
            last_error = str(exc)
        time.sleep(3)
    raise AssertionError(f"vector-retriever did not become ready: {last_error}")


def request_json(
    method: str,
    path: str,
    *,
    payload: Any | None = None,
    params: dict[str, Any] | None = None,
    timeout: int = 60,
) -> tuple[requests.Response, Any]:
    base_url = NOTEBOOK_STATE["base_url"]
    if not base_url:
        raise RuntimeError("Stack is not running yet. Start it before issuing API calls.")

    response = requests.request(
        method=method.upper(),
        url=f"{base_url}{path}",
        json=payload,
        params=params,
        timeout=timeout,
    )
    print(f"{method.upper()} {path} -> {response.status_code}")
    try:
        body = response.json()
    except ValueError:
        body = response.text
    if response.status_code >= 400:
        raise AssertionError(f"{method.upper()} {path} failed: {body}")
    return response, body


def show_json(title: str, payload: Any) -> None:
    print(title)
    print(json.dumps(payload, indent=2, sort_keys=True))


def summarize_result_block(result: dict[str, Any]) -> list[str]:
    matched_ids = [item["metadata"]["video_id"] for item in result.get("items", [])]
    summary = {
        "query_id": result.get("query_id"),
        "count": result.get("count"),
        "matched_ids": matched_ids,
        "applied_filters": result.get("applied_filters"),
    }
    show_json("Result summary", summary)
    return matched_ids


def seed_backend_data(env: dict[str, str]) -> None:
    script = (
        "import json\n"
        "from src.common.settings import settings\n"
        "from src.retriever.backend_factory import get_vectordb\n"
        f"expected_backend = {BACKEND!r}\n"
        "assert settings.RETRIEVER_BACKEND == expected_backend, (\n"
        "    f'backend mismatch: expected {expected_backend!r}, got {settings.RETRIEVER_BACKEND!r}'\n"
        ")\n"
        f"docs = json.loads({json.dumps(DOCUMENTS)!r})\n"
        "store = get_vectordb()\n"
        "texts = [item['page_content'] for item in docs]\n"
        "metadatas = [item['metadata'] for item in docs]\n"
        "ids = [item['metadata']['video_id'] for item in docs]\n"
        "try:\n"
        "    store.add_texts(texts=texts, metadatas=metadatas, ids=ids)\n"
        "except TypeError:\n"
        "    store.add_texts(texts=texts, metadatas=metadatas)\n"
        "print('seeded', len(docs))\n"
    )
    run_compose(["exec", "-T", "vector-retriever", "python", "-c", script], env)


def verify_seed_visible(expected_count: int) -> dict[str, Any]:
    payload = [
        {
            "query_id": "seed-verification",
            "query": QUERY_TEXT,
            "top_k": 100,
        }
    ]
    _, body = request_json("POST", "/query", payload=payload)
    assert not body["errors"], body["errors"]
    result = body["results"][0]
    matched_ids = summarize_result_block(result)
    assert result["count"] >= expected_count, (
        f"expected at least {expected_count} visible documents, got {result['count']}"
    )
    print(f"Seed verification matched IDs: {matched_ids}")
    return body


def run_single_query(
    payload: dict[str, Any],
    *,
    expected_ids: set[str] | None = None,
    require_compiled_filter: bool = False,
) -> dict[str, Any]:
    _, body = request_json("POST", "/query", payload=[payload])
    assert not body["errors"], body["errors"]
    result = body["results"][0]
    matched_ids = set(summarize_result_block(result))
    if expected_ids is not None:
        assert matched_ids == expected_ids, (
            f"expected {sorted(expected_ids)}, got {sorted(matched_ids)}"
        )
    if require_compiled_filter:
        compiled = result["applied_filters"].get("compiled_backend_filter")
        assert compiled is not None, "Expected compiled_backend_filter when explain_filters=True"
    return body


def run_filter_case(case_name: str) -> dict[str, Any]:
    case = deepcopy(FILTER_CASE_LOOKUP[case_name])
    payload = {
        "query_id": case_name,
        "query": QUERY_TEXT,
        "top_k": 100,
        "explain_filters": True,
    }
    payload.update(case["payload"])
    print(f"Running filter case: {case_name}")
    print(f"Expected IDs: {sorted(case['expected_ids'])}")
    return run_single_query(payload, expected_ids=set(case["expected_ids"]))


def run_batch_query_demo() -> dict[str, Any]:
    payload = [
        {
            "query_id": "batch-q1",
            "query": QUERY_TEXT,
            "where": {"field": "video_id", "op": "eq", "value": "vid-001"},
            "top_k": 10,
        },
        {
            "query_id": "batch-q2",
            "query": QUERY_TEXT,
            "where": {"field": "video_id", "op": "eq", "value": "vid-002"},
            "top_k": 10,
        },
    ]
    _, body = request_json("POST", "/query", payload=payload)
    assert not body["errors"], body["errors"]
    assert len(body["results"]) == 2, body
    ids_q1 = {item["metadata"]["video_id"] for item in body["results"][0]["items"]}
    ids_q2 = {item["metadata"]["video_id"] for item in body["results"][1]["items"]}
    assert ids_q1 == {"vid-001"}, ids_q1
    assert ids_q2 == {"vid-002"}, ids_q2
    show_json(
        "Batch query summary",
        {
            "request_id": body["request_id"],
            "result_count": len(body["results"]),
            "errors": body["errors"],
            "matched_ids": {
                body["results"][0]["query_id"]: sorted(ids_q1),
                body["results"][1]["query_id"]: sorted(ids_q2),
            },
        },
    )
    return body


def run_explain_filters_demo() -> dict[str, Any]:
    payload = {
        "query_id": "explain-test",
        "query": QUERY_TEXT,
        "where": {"field": "frame_number", "op": "gte", "value": 50},
        "top_k": 10,
        "explain_filters": True,
    }
    return run_single_query(
        payload,
        expected_ids={"vid-005", "vid-006"},
        require_compiled_filter=True,
    )


def run_top_k_demo() -> dict[str, Any]:
    payload = {
        "query_id": "topk-limit",
        "query": QUERY_TEXT,
        "top_k": 2,
    }
    _, body = request_json("POST", "/query", payload=[payload])
    assert not body["errors"], body["errors"]
    result = body["results"][0]
    assert result["count"] == 2, result
    assert len(result["items"]) == 2, result
    summarize_result_block(result)
    return body


def start_stack() -> dict[str, Any]:
    env = build_env()
    NOTEBOOK_STATE["env"] = env
    NOTEBOOK_STATE["base_url"] = f"http://localhost:{env['VECTOR_RETRIEVER_HOST_PORT']}"
    ensure_ov_models_volume()
    run_compose(["up", "-d", "--build"], env)
    print(f"Base URL: {NOTEBOOK_STATE['base_url']}")
    print(f"Index name: {env['INDEX_NAME']}")
    return NOTEBOOK_STATE


def restart_vector_retriever(env: dict[str, str]) -> None:
    run_compose(["restart", "vector-retriever"], env)
    wait_for_ready(NOTEBOOK_STATE["base_url"])


def compose_ps() -> subprocess.CompletedProcess[str]:
    env = NOTEBOOK_STATE["env"]
    if env is None:
        raise RuntimeError("Stack is not running yet.")
    return run_compose(["ps"], env)


def teardown_stack(remove_volumes: bool = True) -> None:
    env = NOTEBOOK_STATE["env"]
    if env is None:
        print("Stack already torn down.")
        return
    args = ["down"]
    if remove_volumes:
        args.append("-v")
    args.append("--remove-orphans")
    try:
        run_compose(args, env)
    finally:
        NOTEBOOK_STATE["env"] = None
        NOTEBOOK_STATE["base_url"] = None
        print("Notebook state cleared.")


def run_image_query_demo() -> dict[str, Any]:
    """Demonstrate image-based query using a base64-encoded 1x1 red PNG."""
    # 1x1 red PNG (87 bytes) — self-contained, no network dependency.
    RED_1X1_PNG_B64 = (
        "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR4"
        "nGP4z8BQDwAEgAF/pooBPQAAAABJRU5ErkJggg=="
    )

    # --- image_base64 query ---
    print("Testing image_base64 query...")
    payload_b64 = [
        {
            "query_id": "image-base64-test",
            "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64},
            "top_k": 5,
        }
    ]
    _, body = request_json("POST", "/query", payload=payload_b64)
    assert not body["errors"], body["errors"]
    result = body["results"][0]
    assert result["query_id"] == "image-base64-test"
    assert result["query"] == "[image_base64]", f"unexpected query label: {result['query']}"
    assert result["count"] >= 1, f"expected at least 1 match, got {result['count']}"
    summarize_result_block(result)

    # --- image query with where filter ---
    print("\nTesting image query with where filter (video_id=vid-001)...")
    payload_filtered = [
        {
            "query_id": "image-filtered-test",
            "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64},
            "where": {"field": "video_id", "op": "eq", "value": "vid-001"},
            "top_k": 10,
        }
    ]
    _, body = request_json("POST", "/query", payload=payload_filtered)
    assert not body["errors"], body["errors"]
    result = body["results"][0]
    matched_ids = {item["metadata"]["video_id"] for item in result["items"]}
    assert matched_ids <= {"vid-001"}, f"filter should restrict to vid-001, got {matched_ids}"
    summarize_result_block(result)

    # --- mutual exclusivity: both query and image should fail ---
    print("\nTesting mutual exclusivity (query + image should return 422)...")
    payload_invalid = [
        {
            "query_id": "invalid-both",
            "query": "person near car",
            "image": {"type": "image_base64", "image_base64": RED_1X1_PNG_B64},
        }
    ]
    response = requests.post(
        f"{NOTEBOOK_STATE['base_url']}/query",
        json=payload_invalid,
        timeout=60,
    )
    assert response.status_code == 422, (
        f"expected 422 for mutual exclusivity violation, got {response.status_code}"
    )
    print(f"POST /query -> {response.status_code} (correctly rejected)")
    show_json("Validation error", response.json())

    return body


## Start the VDMS compose stack

This launches the same base and VDMS overlay compose files used by the automated functional suite.


In [12]:
start_stack()


Seeded OV models volume: docker_ov-models
$ docker compose -f docker/compose.yaml -f docker/compose.vdms.yaml up -d --build
#1 [internal] load local bake definitions
#1 reading from stdin 932B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.10kB done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.12.13-slim
#3 DONE 1.6s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [1/8] FROM docker.io/library/python:3.12.13-slim@sha256:090ba77e2958f6af52a5341f788b50b032dd4ca28377d2893dcf1ecbdfdfe203
#5 resolve docker.io/library/python:3.12.13-slim@sha256:090ba77e2958f6af52a5341f788b50b032dd4ca28377d2893dcf1ecbdfdfe203 0.0s done
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 17.99kB done
#6 DONE 0.0s

#7 [2/8] RUN groupadd --gid 1000 appuser &&     useradd --uid 1000 --gid appuser --shell /bin/bash --create-home appuser
#7 CACHED

#8 [4/8] COPY pyproject.toml REA

time="2026-05-26T14:15:17+05:30" level=warning msg="The \"REGISTRY\" variable is not set. Defaulting to a blank string."
 Image vector-retriever-vdms:latest Building 
 Image vector-retriever-vdms:latest Built 
 Network docker_vr_network Creating 
 Network docker_vr_network Created 
 Volume docker_vdms-db Creating 
 Volume docker_vdms-db Created 
time="2026-05-26T14:15:20+05:30" level=warning msg="volume \"docker_ov-models\" already exists but was not created by Docker Compose. Use `external: true` to use an existing volume"
 Volume docker_data-prep Creating 
 Volume docker_data-prep Created 
 Container multimodal-embedding-serving Creating 
 Container docker-vdms-vector-db-1 Creating 
 Container docker-vdms-vector-db-1 Created 
 Container multimodal-embedding-serving Created 
 Container docker-vector-retriever-1 Creating 
 Container docker-vector-retriever-1 Created 
 Container multimodal-embedding-serving Starting 
 Container docker-vdms-vector-db-1 Starting 
 Container multimodal-emb

{'backend': 'vdms',
 'env': {'HTTPS_PROXY': 'http://proxy-dmz.intel.com:912',
  'no_proxy': 'intel.com,.intel.com,localhost,127.0.0.1,10.0.0.0/8,10.223.22.41',
  'USER': 'intel',
  'SSH_CLIENT': '10.125.130.51 49677 22',
  'XDG_SESSION_TYPE': 'tty',
  'SHLVL': '0',
  'SOCKS_PROXY': 'http://proxy-dmz.intel.com:1080',
  'HOME': '/home/intel',
  'NO_PROXY': 'intel.com,.intel.com,localhost,127.0.0.1,10.0.0.0/8,10.223.22.41',
  'DBUS_SESSION_BUS_ADDRESS': 'unix:path=/run/user/1000/bus',
  'https_proxy': 'http://proxy-dmz.intel.com:912',
  'LOGNAME': 'intel',
  'http_proxy': 'http://proxy-dmz.intel.com:912',
  '_': '/home/intel/workbench/.venv/bin/python',
  'XDG_SESSION_CLASS': 'user',
  'socks_proxy': 'http://proxy-dmz.intel.com:1080',
  'XDG_SESSION_ID': '23430',
  'PATH': '/home/intel/workbench/.venv/bin:/home/intel/.vscode-server/bin/f6cfa2ea2403534de03f069bdf160d06451ed282/bin/remote-cli:/home/intel/.local/bin:/home/intel/.nvm/versions/node/v24.14.0/bin:/usr/local/sbin:/usr/local/bin:/

## Wait for readiness, seed deterministic documents, and restart after seeding

The functional tests intentionally restart `vector-retriever` after seeding for VDMS.
That restart is important because VDMS metadata field discovery happens during initialization,
and a restart refreshes the collection properties so newly seeded custom fields are visible in search results.


In [6]:
wait_for_ready(NOTEBOOK_STATE["base_url"])
seed_backend_data(NOTEBOOK_STATE["env"])
print("Restarting vector-retriever to refresh VDMS collection metadata after seeding...")
restart_vector_retriever(NOTEBOOK_STATE["env"])


Service ready at http://localhost:6101
$ docker compose -f docker/compose.yaml -f docker/compose.vdms.yaml exec -T vector-retriever python -c import json
from src.common.settings import settings
from src.retriever.backend_factory import get_vectordb
expected_backend = 'vdms'
assert settings.RETRIEVER_BACKEND == expected_backend, (
    f'backend mismatch: expected {expected_backend!r}, got {settings.RETRIEVER_BACKEND!r}'
)
docs = json.loads('[{"page_content": "red car at intersection", "metadata": {"video_id": "vid-001", "camera_zone": "north", "description": "red car at intersection", "prefix": "alpha-route", "frame_number": 10, "bucket_name": "cam-a", "tags": ["traffic", "vehicle", "red"], "created_at": "2026-03-10T10:00:00+00:00", "optional_note": "doc has note"}}, {"page_content": "blue bus near station", "metadata": {"video_id": "vid-002", "camera_zone": "south", "description": "blue bus near station", "prefix": "beta-route", "frame_number": 20, "bucket_name": "cam-b", "tags": ["tr

time="2026-05-26T14:05:05+05:30" level=warning msg="The \"REGISTRY\" variable is not set. Defaulting to a blank string."
2026-05-26 08:35:05,932 - DEBUG - Retrieving vector store for backend 'vdms'
2026-05-26 08:35:05,932 - DEBUG - Loading backend callable 'get_vectordb' from 'src.retriever.backends.vdms.backend'
2026-05-26 08:35:06,433 - INFO - Initializing VDMS backend for collection 'vr_functional_vdms_af28e7d4' at 'vdms-vector-db:55555'
2026-05-26 08:35:06,435 - DEBUG - Starting new HTTP connection (1): multimodal-embedding-serving:8000
2026-05-26 08:35:06,449 - DEBUG - http://multimodal-embedding-serving:8000 "POST /embeddings HTTP/1.1" 200 10838
2026-05-26 08:35:06,449 - DEBUG - Resolved VDMS embedding dimension: 512
2026-05-26 08:35:06,498 - INFO - Descriptor set vr_functional_vdms_af28e7d4 exists
2026-05-26 08:35:06,498 - DEBUG - Retriever backend 'vdms' initialized successfully
2026-05-26 08:35:06,499 - DEBUG - Starting new HTTP connection (1): multimodal-embedding-serving:800

Service ready at http://localhost:6101


## Verify that the seed is visible

Before exploring specific filters, confirm that the seeded fixture corpus is queryable through `/query`.


In [15]:
verify_seed_visible(expected_count=len(DOCUMENTS))


POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": null,
    "filters": null,
    "normalized_where": null,
    "tags": null,
    "time_filter": null,
    "warnings": null
  },
  "count": 6,
  "matched_ids": [
    "vid-005",
    "vid-004",
    "vid-003",
    "vid-006",
    "vid-001",
    "vid-002"
  ],
  "query_id": "seed-verification"
}
Seed verification matched IDs: ['vid-005', 'vid-004', 'vid-003', 'vid-006', 'vid-001', 'vid-002']


{'request_id': 'fde448f7-3061-4b3a-b7dd-fd68e8b661e0',
 'results': [{'query_id': 'seed-verification',
   'query': 'traffic light malfunctio',
   'count': 6,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.6137043238,
     'metadata': {'bucket_name': 'cam-d',
      'camera_zone': 'west',
      'created_at': '2026-03-25T14:15:00+00:00',
      'description': 'delivery truck unloading',
      'frame_number': 40,
      'prefix': 'gamma-log',
      'tags': 'logistics,vehicle',
      'video_id': 'vid-004'},
     'page_content': 'delivery truck unloading'},
    {'score': 0.60422194,
     'metadata': {'bucket_n

## Endpoint check: `/health`

`/health` is the lightweight liveness signal. It should return `status="ok"`.


In [16]:
_, health_payload = request_json("GET", "/health", timeout=15)
assert health_payload["status"] == "ok", health_payload
show_json("/health payload", health_payload)


GET /health -> 200
/health payload
{
  "status": "ok",
  "timestamp": "2026-05-25T09:55:40.958555Z"
}


## Endpoint check: `/ready`

`/ready` validates runtime dependencies. After startup and the post-seed restart, it should report `status="ready"`.


In [17]:
_, ready_payload = request_json("GET", "/ready", timeout=15)
assert ready_payload["status"] == "ready", ready_payload
show_json("/ready payload", ready_payload)


GET /ready -> 200
/ready payload
{
  "status": "ready",
  "timestamp": "2026-05-25T09:55:43.468789Z"
}


## Endpoint check: `/capabilities/filters`

This endpoint advertises the supported filter grammar and confirms that the active backend is `vdms`.


In [18]:
_, capabilities_payload = request_json("GET", "/capabilities/filters", timeout=15)
assert capabilities_payload["active_backend"] == BACKEND, capabilities_payload
vdms_capabilities = next(
    backend for backend in capabilities_payload["backends"] if backend["backend"] == BACKEND
)
show_json(
    "VDMS filter capabilities snapshot",
    {
        "active_backend": capabilities_payload["active_backend"],
        "supported_backends": [backend["backend"] for backend in capabilities_payload["backends"]],
        "vdms_supported_operators": vdms_capabilities["supported_operators"],
        "vdms_pushdown_operators": vdms_capabilities["pushdown_operators"],
        "known_fields": vdms_capabilities["known_fields"],
    },
)


GET /capabilities/filters -> 200
VDMS filter capabilities snapshot
{
  "active_backend": "vdms",
  "known_fields": {
    "created_at": "datetime",
    "tags": "array<string>"
  },
  "supported_backends": [
    "faiss",
    "milvus",
    "pgvector",
    "vdms"
  ],
  "vdms_pushdown_operators": [
    "gte"
  ],
  "vdms_supported_operators": [
    "eq",
    "in",
    "contains",
    "starts_with",
    "gt",
    "gte",
    "lt",
    "lte",
    "between",
    "contains_any",
    "contains_all",
    "exists",
    "missing"
  ]
}


## Query behavior: batch query

This mirrors `assert_batch_query` from the functional tests and confirms that a single `/query`
request can carry multiple independent query blocks.


In [11]:
run_batch_query_demo()


POST /query -> 200
Batch query summary
{
  "errors": [],
  "matched_ids": {
    "batch-q1": [
      "vid-001"
    ],
    "batch-q2": [
      "vid-002"
    ]
  },
  "request_id": "3cfd3097-9a62-4dea-a9cc-82687dff169d",
  "result_count": 2
}


{'request_id': '3cfd3097-9a62-4dea-a9cc-82687dff169d',
 'results': [{'query_id': 'batch-q1',
   'query': 'traffic light malfunctio',
   'count': 1,
   'items': [{'score': 0.4819686711,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north',
      'created_at': '2026-03-10T10:00:00+00:00',
      'description': 'red car at intersection',
      'frame_number': 10,
      'optional_note': 'doc has note',
      'prefix': 'alpha-route',
      'tags': 'traffic,vehicle,red',
      'video_id': 'vid-001'},
     'page_content': 'red car at intersection'}],
   'applied_filters': {'tags': None,
    'time_filter': None,
    'filters': None,
    'normalized_where': {'field': 'video_id',
     'op': 'eq',
     'value': 'vid-001',
     'all': None,
     'any': None,
     'not': None},
    'warnings': ['Predicate video_id:eq is not backend-pushdown compatible and will be evaluated in fallback.'],
    'compiled_backend_filter': None,
    'dropped_or_rewritten_clauses': None}},
  {'query_id'

## Query behavior: `explain_filters`

This call requests filter explanation metadata and checks that VDMS returns a compiled backend filter description.


In [12]:
run_explain_filters_demo()


POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": {
      "frame_number": [
        ">=",
        50
      ]
    },
    "dropped_or_rewritten_clauses": [],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "frame_number",
      "not": null,
      "op": "gte",
      "value": 50
    },
    "tags": null,
    "time_filter": null,
    "warnings": null
  },
  "count": 2,
  "matched_ids": [
    "vid-005",
    "vid-006"
  ],
  "query_id": "explain-test"
}


{'request_id': '83d2d2c8-1bd3-4d7c-86e6-6837c03aa19e',
 'results': [{'query_id': 'explain-test',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.5379798412,
     'metadata': {'bucket_name': 'cam-e',
      'camera_zone': 'south-west',
      'created_at': '2026-04-05T20:10:00+00:00',
      'description': 'empty intersection at night',
      'frame_number': 60,
      'prefix': 'delta-night',
      'tags': 'night,traffic',
      'video_id': 'vid-006'},
     'page_content': 'empty intersection at night'}],
   'applied_filters': {'tags': None,
    'time_

## Query behavior: `top_k` limiting

This demonstrates that `/query` enforces `top_k` even when the fixture corpus contains more matching documents.


In [13]:
run_top_k_demo()


POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": null,
    "filters": null,
    "normalized_where": null,
    "tags": null,
    "time_filter": null,
    "warnings": null
  },
  "count": 2,
  "matched_ids": [
    "vid-005",
    "vid-004"
  ],
  "query_id": "topk-limit"
}


{'request_id': '8eb2d079-523e-45b1-b07a-8fccf4c1c523',
 'results': [{'query_id': 'topk-limit',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.6137043238,
     'metadata': {'bucket_name': 'cam-d',
      'camera_zone': 'west',
      'created_at': '2026-03-25T14:15:00+00:00',
      'description': 'delivery truck unloading',
      'frame_number': 40,
      'prefix': 'gamma-log',
      'tags': 'logistics,vehicle',
      'video_id': 'vid-004'},
     'page_content': 'delivery truck unloading'}],
   'applied_filters': {'tags': None,
    'time_filter': Non

## Query behavior: image query

This demonstrates the image query modality introduced alongside the existing text query.
An image can be provided as `image_base64` or `image_url` instead of a text `query`.
The two modalities are mutually exclusive — providing both returns a `422` validation error.

The test uses a tiny 1×1 red PNG encoded in base64 so it works without network access.
The embedding service generates a vector from the image and vector search runs against the
seeded fixture corpus.

In [14]:
run_image_query_demo()

Testing image_base64 query...


POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": null,
    "filters": null,
    "normalized_where": null,
    "tags": null,
    "time_filter": null,
    "warnings": null
  },
  "count": 5,
  "matched_ids": [
    "vid-001",
    "vid-005",
    "vid-003",
    "vid-004",
    "vid-006"
  ],
  "query_id": "image-base64-test"
}

Testing image query with where filter (video_id=vid-001)...
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": null,
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "video_id",
      "not": null,
      "op": "eq",
      "value": "vid-001"
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate video_id:eq is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 1,
  "matched_ids": [
    "vid-

{'request_id': '667d6181-70b4-435d-bb98-97a2d4afec0b',
 'results': [{'query_id': 'image-filtered-test',
   'query': '[image_base64]',
   'count': 1,
   'items': [{'score': 0.2077488452,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north',
      'created_at': '2026-03-10T10:00:00+00:00',
      'description': 'red car at intersection',
      'frame_number': 10,
      'optional_note': 'doc has note',
      'prefix': 'alpha-route',
      'tags': 'traffic,vehicle,red',
      'video_id': 'vid-001'},
     'page_content': 'red car at intersection'}],
   'applied_filters': {'tags': None,
    'time_filter': None,
    'filters': None,
    'normalized_where': {'field': 'video_id',
     'op': 'eq',
     'value': 'vid-001',
     'all': None,
     'any': None,
     'not': None},
    'warnings': ['Predicate video_id:eq is not backend-pushdown compatible and will be evaluated in fallback.'],
    'compiled_backend_filter': None,
    'dropped_or_rewritten_clauses': None}}],
 'errors': 

## Filter case: `op_eq`

This cell mirrors the `op_eq` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-001']`

**Payload overlay:**

```json
{
  "where": {
    "field": "video_id",
    "op": "eq",
    "value": "vid-001"
  }
}
```


In [15]:
run_filter_case("op_eq")


Running filter case: op_eq
Expected IDs: ['vid-001']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "video_id:eq"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "video_id",
      "not": null,
      "op": "eq",
      "value": "vid-001"
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate video_id:eq is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 1,
  "matched_ids": [
    "vid-001"
  ],
  "query_id": "op_eq"
}


{'request_id': 'e908a3c4-6985-4927-aea1-63798cf70e2b',
 'results': [{'query_id': 'op_eq',
   'query': 'traffic light malfunctio',
   'count': 1,
   'items': [{'score': 0.4819686711,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north',
      'created_at': '2026-03-10T10:00:00+00:00',
      'description': 'red car at intersection',
      'frame_number': 10,
      'optional_note': 'doc has note',
      'prefix': 'alpha-route',
      'tags': 'traffic,vehicle,red',
      'video_id': 'vid-001'},
     'page_content': 'red car at intersection'}],
   'applied_filters': {'tags': None,
    'time_filter': None,
    'filters': None,
    'normalized_where': {'field': 'video_id',
     'op': 'eq',
     'value': 'vid-001',
     'all': None,
     'any': None,
     'not': None},
    'warnings': ['Predicate video_id:eq is not backend-pushdown compatible and will be evaluated in fallback.'],
    'compiled_backend_filter': None,
    'dropped_or_rewritten_clauses': ['video_id:eq']}}],
 'er

## Filter case: `op_in`

This cell mirrors the `op_in` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-001', 'vid-003']`

**Payload overlay:**

```json
{
  "where": {
    "field": "camera_zone",
    "op": "in",
    "value": [
      "north",
      "east"
    ]
  }
}
```


In [16]:
run_filter_case("op_in")


Running filter case: op_in
Expected IDs: ['vid-001', 'vid-003']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "camera_zone:in"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "camera_zone",
      "not": null,
      "op": "in",
      "value": [
        "north",
        "east"
      ]
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate camera_zone:in is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 2,
  "matched_ids": [
    "vid-003",
    "vid-001"
  ],
  "query_id": "op_in"
}


{'request_id': 'c1a336fd-bd18-4771-b147-94bb2fade0e7',
 'results': [{'query_id': 'op_in',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.60422194,
     'metadata': {'bucket_name': 'cam-c',
      'camera_zone': 'east',
      'created_at': '2026-03-20T08:30:00+00:00',
      'description': 'person with bicycle crossing',
      'frame_number': 30,
      'optional_note': 'pedestrian event',
      'prefix': 'alpha-bike',
      'tags': 'pedestrian,bicycle',
      'video_id': 'vid-003'},
     'page_content': 'person with bicycle crossing'},
    {'score': 0.4819686711,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north',
      'created_at': '2026-03-10T10:00:00+00:00',
      'description': 'red car at intersection',
      'frame_number': 10,
      'optional_note': 'doc has note',
      'prefix': 'alpha-route',
      'tags': 'traffic,vehicle,red',
      'video_id': 'vid-001'},
     'page_content': 'red car at intersection'}],
   'applied_filter

## Filter case: `op_contains`

This cell mirrors the `op_contains` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-001', 'vid-006']`

**Payload overlay:**

```json
{
  "where": {
    "field": "description",
    "op": "contains",
    "value": "intersection"
  }
}
```


In [17]:
run_filter_case("op_contains")


Running filter case: op_contains
Expected IDs: ['vid-001', 'vid-006']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "description:contains"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "description",
      "not": null,
      "op": "contains",
      "value": "intersection"
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate description:contains is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 2,
  "matched_ids": [
    "vid-006",
    "vid-001"
  ],
  "query_id": "op_contains"
}


{'request_id': '4f1e4d7a-968f-4e33-b81c-ecbeb71d70da',
 'results': [{'query_id': 'op_contains',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.5379798412,
     'metadata': {'bucket_name': 'cam-e',
      'camera_zone': 'south-west',
      'created_at': '2026-04-05T20:10:00+00:00',
      'description': 'empty intersection at night',
      'frame_number': 60,
      'prefix': 'delta-night',
      'tags': 'night,traffic',
      'video_id': 'vid-006'},
     'page_content': 'empty intersection at night'},
    {'score': 0.4819686711,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north',
      'created_at': '2026-03-10T10:00:00+00:00',
      'description': 'red car at intersection',
      'frame_number': 10,
      'optional_note': 'doc has note',
      'prefix': 'alpha-route',
      'tags': 'traffic,vehicle,red',
      'video_id': 'vid-001'},
     'page_content': 'red car at intersection'}],
   'applied_filters': {'tags': None,
    'time_filter

## Filter case: `op_starts_with`

This cell mirrors the `op_starts_with` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-001', 'vid-003', 'vid-005']`

**Payload overlay:**

```json
{
  "where": {
    "field": "prefix",
    "op": "starts_with",
    "value": "alpha"
  }
}
```


In [18]:
run_filter_case("op_starts_with")


Running filter case: op_starts_with
Expected IDs: ['vid-001', 'vid-003', 'vid-005']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "prefix:starts_with"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "prefix",
      "not": null,
      "op": "starts_with",
      "value": "alpha"
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate prefix:starts_with is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 3,
  "matched_ids": [
    "vid-005",
    "vid-003",
    "vid-001"
  ],
  "query_id": "op_starts_with"
}


{'request_id': '85b06117-7b31-4f83-b865-e54b98ffc0f4',
 'results': [{'query_id': 'op_starts_with',
   'query': 'traffic light malfunctio',
   'count': 3,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.60422194,
     'metadata': {'bucket_name': 'cam-c',
      'camera_zone': 'east',
      'created_at': '2026-03-20T08:30:00+00:00',
      'description': 'person with bicycle crossing',
      'frame_number': 30,
      'optional_note': 'pedestrian event',
      'prefix': 'alpha-bike',
      'tags': 'pedestrian,bicycle',
      'video_id': 'vid-003'},
     'page_content': 'person with bicycle crossing'},
    {

## Filter case: `op_gt`

This cell mirrors the `op_gt` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-005', 'vid-006']`

**Payload overlay:**

```json
{
  "where": {
    "field": "frame_number",
    "op": "gt",
    "value": 40
  }
}
```


In [19]:
run_filter_case("op_gt")


Running filter case: op_gt
Expected IDs: ['vid-005', 'vid-006']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "frame_number:gt"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "frame_number",
      "not": null,
      "op": "gt",
      "value": 40
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate frame_number:gt is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 2,
  "matched_ids": [
    "vid-005",
    "vid-006"
  ],
  "query_id": "op_gt"
}


{'request_id': '8b29e1f7-6086-4619-8be3-971b61688fe3',
 'results': [{'query_id': 'op_gt',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.5379798412,
     'metadata': {'bucket_name': 'cam-e',
      'camera_zone': 'south-west',
      'created_at': '2026-04-05T20:10:00+00:00',
      'description': 'empty intersection at night',
      'frame_number': 60,
      'prefix': 'delta-night',
      'tags': 'night,traffic',
      'video_id': 'vid-006'},
     'page_content': 'empty intersection at night'}],
   'applied_filters': {'tags': None,
    'time_filter'

## Filter case: `op_gte`

This cell mirrors the `op_gte` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-004', 'vid-005', 'vid-006']`

**Payload overlay:**

```json
{
  "where": {
    "field": "frame_number",
    "op": "gte",
    "value": 40
  }
}
```


In [20]:
run_filter_case("op_gte")


Running filter case: op_gte
Expected IDs: ['vid-004', 'vid-005', 'vid-006']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": {
      "frame_number": [
        ">=",
        40
      ]
    },
    "dropped_or_rewritten_clauses": [],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "frame_number",
      "not": null,
      "op": "gte",
      "value": 40
    },
    "tags": null,
    "time_filter": null,
    "warnings": null
  },
  "count": 3,
  "matched_ids": [
    "vid-005",
    "vid-004",
    "vid-006"
  ],
  "query_id": "op_gte"
}


{'request_id': '887a12a1-bf26-49f7-9397-a2cfb04b8d05',
 'results': [{'query_id': 'op_gte',
   'query': 'traffic light malfunctio',
   'count': 3,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.6137043238,
     'metadata': {'bucket_name': 'cam-d',
      'camera_zone': 'west',
      'created_at': '2026-03-25T14:15:00+00:00',
      'description': 'delivery truck unloading',
      'frame_number': 40,
      'prefix': 'gamma-log',
      'tags': 'logistics,vehicle',
      'video_id': 'vid-004'},
     'page_content': 'delivery truck unloading'},
    {'score': 0.5379798412,
     'metadata': {'bucket_name': 'ca

## Filter case: `op_lt`

This cell mirrors the `op_lt` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-001', 'vid-002']`

**Payload overlay:**

```json
{
  "where": {
    "field": "frame_number",
    "op": "lt",
    "value": 30
  }
}
```


In [21]:
run_filter_case("op_lt")


Running filter case: op_lt
Expected IDs: ['vid-001', 'vid-002']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "frame_number:lt"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "frame_number",
      "not": null,
      "op": "lt",
      "value": 30
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate frame_number:lt is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 2,
  "matched_ids": [
    "vid-001",
    "vid-002"
  ],
  "query_id": "op_lt"
}


{'request_id': 'ee2e3227-fd42-4b15-ab0e-de0a9fd6a40d',
 'results': [{'query_id': 'op_lt',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.4819686711,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north',
      'created_at': '2026-03-10T10:00:00+00:00',
      'description': 'red car at intersection',
      'frame_number': 10,
      'optional_note': 'doc has note',
      'prefix': 'alpha-route',
      'tags': 'traffic,vehicle,red',
      'video_id': 'vid-001'},
     'page_content': 'red car at intersection'},
    {'score': 0.3594945371,
     'metadata': {'bucket_name': 'cam-b',
      'camera_zone': 'south',
      'created_at': '2026-03-15T12:00:00+00:00',
      'description': 'blue bus near station',
      'frame_number': 20,
      'prefix': 'beta-route',
      'tags': 'traffic,vehicle,bus',
      'video_id': 'vid-002'},
     'page_content': 'blue bus near station'}],
   'applied_filters': {'tags': None,
    'time_filter': None,
    'filt

## Filter case: `op_lte`

This cell mirrors the `op_lte` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-001', 'vid-002', 'vid-003']`

**Payload overlay:**

```json
{
  "where": {
    "field": "frame_number",
    "op": "lte",
    "value": 30
  }
}
```


In [22]:
run_filter_case("op_lte")


Running filter case: op_lte
Expected IDs: ['vid-001', 'vid-002', 'vid-003']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "frame_number:lte"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "frame_number",
      "not": null,
      "op": "lte",
      "value": 30
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate frame_number:lte is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 3,
  "matched_ids": [
    "vid-003",
    "vid-001",
    "vid-002"
  ],
  "query_id": "op_lte"
}


{'request_id': '637fb566-3c0b-4bba-b8a2-d4b0730b921c',
 'results': [{'query_id': 'op_lte',
   'query': 'traffic light malfunctio',
   'count': 3,
   'items': [{'score': 0.60422194,
     'metadata': {'bucket_name': 'cam-c',
      'camera_zone': 'east',
      'created_at': '2026-03-20T08:30:00+00:00',
      'description': 'person with bicycle crossing',
      'frame_number': 30,
      'optional_note': 'pedestrian event',
      'prefix': 'alpha-bike',
      'tags': 'pedestrian,bicycle',
      'video_id': 'vid-003'},
     'page_content': 'person with bicycle crossing'},
    {'score': 0.4819686711,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north',
      'created_at': '2026-03-10T10:00:00+00:00',
      'description': 'red car at intersection',
      'frame_number': 10,
      'optional_note': 'doc has note',
      'prefix': 'alpha-route',
      'tags': 'traffic,vehicle,red',
      'video_id': 'vid-001'},
     'page_content': 'red car at intersection'},
    {'score': 0.35

## Filter case: `op_between`

This cell mirrors the `op_between` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-002', 'vid-003', 'vid-004']`

**Payload overlay:**

```json
{
  "where": {
    "field": "frame_number",
    "op": "between",
    "value": [
      20,
      40
    ]
  }
}
```


In [23]:
run_filter_case("op_between")


Running filter case: op_between
Expected IDs: ['vid-002', 'vid-003', 'vid-004']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "frame_number:between"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "frame_number",
      "not": null,
      "op": "between",
      "value": [
        20,
        40
      ]
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate frame_number:between is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 3,
  "matched_ids": [
    "vid-004",
    "vid-003",
    "vid-002"
  ],
  "query_id": "op_between"
}


{'request_id': 'd2f32fd9-3c95-488b-b03f-b7002baf2a1c',
 'results': [{'query_id': 'op_between',
   'query': 'traffic light malfunctio',
   'count': 3,
   'items': [{'score': 0.6137043238,
     'metadata': {'bucket_name': 'cam-d',
      'camera_zone': 'west',
      'created_at': '2026-03-25T14:15:00+00:00',
      'description': 'delivery truck unloading',
      'frame_number': 40,
      'prefix': 'gamma-log',
      'tags': 'logistics,vehicle',
      'video_id': 'vid-004'},
     'page_content': 'delivery truck unloading'},
    {'score': 0.60422194,
     'metadata': {'bucket_name': 'cam-c',
      'camera_zone': 'east',
      'created_at': '2026-03-20T08:30:00+00:00',
      'description': 'person with bicycle crossing',
      'frame_number': 30,
      'optional_note': 'pedestrian event',
      'prefix': 'alpha-bike',
      'tags': 'pedestrian,bicycle',
      'video_id': 'vid-003'},
     'page_content': 'person with bicycle crossing'},
    {'score': 0.3594945371,
     'metadata': {'bucket_na

## Filter case: `op_contains_any`

This cell mirrors the `op_contains_any` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-003', 'vid-005']`

**Payload overlay:**

```json
{
  "where": {
    "field": "tags",
    "op": "contains_any",
    "value": [
      "bicycle",
      "signal"
    ]
  }
}
```


In [24]:
run_filter_case("op_contains_any")


Running filter case: op_contains_any
Expected IDs: ['vid-003', 'vid-005']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "tags:contains_any"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "tags",
      "not": null,
      "op": "contains_any",
      "value": [
        "bicycle",
        "signal"
      ]
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate tags:contains_any is evaluated in fallback path (list-typed field)."
    ]
  },
  "count": 2,
  "matched_ids": [
    "vid-005",
    "vid-003"
  ],
  "query_id": "op_contains_any"
}


{'request_id': '356f4f79-b7b1-4f64-9779-04618c782af8',
 'results': [{'query_id': 'op_contains_any',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.60422194,
     'metadata': {'bucket_name': 'cam-c',
      'camera_zone': 'east',
      'created_at': '2026-03-20T08:30:00+00:00',
      'description': 'person with bicycle crossing',
      'frame_number': 30,
      'optional_note': 'pedestrian event',
      'prefix': 'alpha-bike',
      'tags': 'pedestrian,bicycle',
      'video_id': 'vid-003'},
     'page_content': 'person with bicycle crossing'}],
   

## Filter case: `op_contains_all`

This cell mirrors the `op_contains_all` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-001', 'vid-002']`

**Payload overlay:**

```json
{
  "where": {
    "field": "tags",
    "op": "contains_all",
    "value": [
      "traffic",
      "vehicle"
    ]
  }
}
```


In [25]:
run_filter_case("op_contains_all")


Running filter case: op_contains_all
Expected IDs: ['vid-001', 'vid-002']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "tags:contains_all"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "tags",
      "not": null,
      "op": "contains_all",
      "value": [
        "traffic",
        "vehicle"
      ]
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate tags:contains_all is evaluated in fallback path (list-typed field)."
    ]
  },
  "count": 2,
  "matched_ids": [
    "vid-001",
    "vid-002"
  ],
  "query_id": "op_contains_all"
}


{'request_id': '2906cc11-02b2-4878-942a-b0a5d6e1f392',
 'results': [{'query_id': 'op_contains_all',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.4819686711,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north',
      'created_at': '2026-03-10T10:00:00+00:00',
      'description': 'red car at intersection',
      'frame_number': 10,
      'optional_note': 'doc has note',
      'prefix': 'alpha-route',
      'tags': 'traffic,vehicle,red',
      'video_id': 'vid-001'},
     'page_content': 'red car at intersection'},
    {'score': 0.3594945371,
     'metadata': {'bucket_name': 'cam-b',
      'camera_zone': 'south',
      'created_at': '2026-03-15T12:00:00+00:00',
      'description': 'blue bus near station',
      'frame_number': 20,
      'prefix': 'beta-route',
      'tags': 'traffic,vehicle,bus',
      'video_id': 'vid-002'},
     'page_content': 'blue bus near station'}],
   'applied_filters': {'tags': None,
    'time_filter': None,

## Filter case: `op_exists`

This cell mirrors the `op_exists` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-001', 'vid-003', 'vid-005']`

**Payload overlay:**

```json
{
  "where": {
    "field": "optional_note",
    "op": "exists"
  }
}
```


In [26]:
run_filter_case("op_exists")


Running filter case: op_exists
Expected IDs: ['vid-001', 'vid-003', 'vid-005']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "optional_note:exists"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "optional_note",
      "not": null,
      "op": "exists",
      "value": null
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate optional_note:exists is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 3,
  "matched_ids": [
    "vid-005",
    "vid-003",
    "vid-001"
  ],
  "query_id": "op_exists"
}


{'request_id': '76d62dc3-9c3a-4e8c-adf3-28b68e62491f',
 'results': [{'query_id': 'op_exists',
   'query': 'traffic light malfunctio',
   'count': 3,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.60422194,
     'metadata': {'bucket_name': 'cam-c',
      'camera_zone': 'east',
      'created_at': '2026-03-20T08:30:00+00:00',
      'description': 'person with bicycle crossing',
      'frame_number': 30,
      'optional_note': 'pedestrian event',
      'prefix': 'alpha-bike',
      'tags': 'pedestrian,bicycle',
      'video_id': 'vid-003'},
     'page_content': 'person with bicycle crossing'},
    {'scor

## Filter case: `op_missing`

This cell mirrors the `op_missing` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-002', 'vid-004', 'vid-006']`

**Payload overlay:**

```json
{
  "where": {
    "field": "optional_note",
    "op": "missing"
  }
}
```


In [27]:
run_filter_case("op_missing")


Running filter case: op_missing
Expected IDs: ['vid-002', 'vid-004', 'vid-006']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "optional_note:missing"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "optional_note",
      "not": null,
      "op": "missing",
      "value": null
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate optional_note:missing is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 3,
  "matched_ids": [
    "vid-004",
    "vid-006",
    "vid-002"
  ],
  "query_id": "op_missing"
}


{'request_id': '83eee5dd-5f1b-42c5-a076-41cd3237602a',
 'results': [{'query_id': 'op_missing',
   'query': 'traffic light malfunctio',
   'count': 3,
   'items': [{'score': 0.6137043238,
     'metadata': {'bucket_name': 'cam-d',
      'camera_zone': 'west',
      'created_at': '2026-03-25T14:15:00+00:00',
      'description': 'delivery truck unloading',
      'frame_number': 40,
      'prefix': 'gamma-log',
      'tags': 'logistics,vehicle',
      'video_id': 'vid-004'},
     'page_content': 'delivery truck unloading'},
    {'score': 0.5379798412,
     'metadata': {'bucket_name': 'cam-e',
      'camera_zone': 'south-west',
      'created_at': '2026-04-05T20:10:00+00:00',
      'description': 'empty intersection at night',
      'frame_number': 60,
      'prefix': 'delta-night',
      'tags': 'night,traffic',
      'video_id': 'vid-006'},
     'page_content': 'empty intersection at night'},
    {'score': 0.3594945371,
     'metadata': {'bucket_name': 'cam-b',
      'camera_zone': 'south

## Filter case: `logical_compound`

This cell mirrors the `logical_compound` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-003', 'vid-004']`

**Payload overlay:**

```json
{
  "where": {
    "all": [
      {
        "field": "frame_number",
        "op": "gte",
        "value": 20
      },
      {
        "any": [
          {
            "field": "camera_zone",
            "op": "eq",
            "value": "east"
          },
          {
            "field": "camera_zone",
            "op": "eq",
            "value": "west"
          }
        ]
      },
      {
        "not": {
          "field": "tags",
          "op": "contains_any",
          "value": [
            "night"
          ]
        }
      }
    ]
  }
}
```


In [28]:
run_filter_case("logical_compound")


Running filter case: logical_compound
Expected IDs: ['vid-003', 'vid-004']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": {
      "frame_number": [
        ">=",
        20
      ]
    },
    "dropped_or_rewritten_clauses": [
      "any(...)",
      "not(...)"
    ],
    "filters": null,
    "normalized_where": {
      "all": [
        {
          "all": null,
          "any": null,
          "field": "frame_number",
          "not": null,
          "op": "gte",
          "value": 20
        },
        {
          "all": null,
          "any": [
            {
              "all": null,
              "any": null,
              "field": "camera_zone",
              "not": null,
              "op": "eq",
              "value": "east"
            },
            {
              "all": null,
              "any": null,
              "field": "camera_zone",
              "not": null,
              "op": "eq",
              "value": "west"
            

{'request_id': 'd00d69cd-1128-4c84-b4f7-926937085b06',
 'results': [{'query_id': 'logical_compound',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.6137043238,
     'metadata': {'bucket_name': 'cam-d',
      'camera_zone': 'west',
      'created_at': '2026-03-25T14:15:00+00:00',
      'description': 'delivery truck unloading',
      'frame_number': 40,
      'prefix': 'gamma-log',
      'tags': 'logistics,vehicle',
      'video_id': 'vid-004'},
     'page_content': 'delivery truck unloading'},
    {'score': 0.60422194,
     'metadata': {'bucket_name': 'cam-c',
      'camera_zone': 'east',
      'created_at': '2026-03-20T08:30:00+00:00',
      'description': 'person with bicycle crossing',
      'frame_number': 30,
      'optional_note': 'pedestrian event',
      'prefix': 'alpha-bike',
      'tags': 'pedestrian,bicycle',
      'video_id': 'vid-003'},
     'page_content': 'person with bicycle crossing'}],
   'applied_filters': {'tags': None,
    'time_fi

## Filter case: `legacy_tags_and_time_filter`

This cell mirrors the `legacy_tags_and_time_filter` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-002', 'vid-005']`

**Payload overlay:**

```json
{
  "tags": [
    "traffic"
  ],
  "time_filter": {
    "end": "2026-04-02T00:00:00+00:00",
    "start": "2026-03-12T00:00:00+00:00"
  }
}
```


In [29]:
run_filter_case("legacy_tags_and_time_filter")


Running filter case: legacy_tags_and_time_filter
Expected IDs: ['vid-002', 'vid-005']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": {
      "created_at": [
        ">=",
        "2026-03-12T00:00:00+00:00",
        "<=",
        "2026-04-02T00:00:00+00:00"
      ]
    },
    "dropped_or_rewritten_clauses": [
      "tags -> where(field='tags', op='contains_any', value=<tags>)",
      "time_filter -> where(field='created_at', op='between', value=[start, end])",
      "tags:contains_any"
    ],
    "filters": null,
    "normalized_where": {
      "all": [
        {
          "all": null,
          "any": null,
          "field": "tags",
          "not": null,
          "op": "contains_any",
          "value": [
            "traffic"
          ]
        },
        {
          "all": null,
          "any": null,
          "field": "created_at",
          "not": null,
          "op": "between",
          "value": [
            "2026-03-12T00:00:00Z

{'request_id': '48711398-1add-4220-ad94-f22119ef1031',
 'results': [{'query_id': 'legacy_tags_and_time_filter',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.3594945371,
     'metadata': {'bucket_name': 'cam-b',
      'camera_zone': 'south',
      'created_at': '2026-03-15T12:00:00+00:00',
      'description': 'blue bus near station',
      'frame_number': 20,
      'prefix': 'beta-route',
      'tags': 'traffic,vehicle,bus',
      'video_id': 'vid-002'},
     'page_content': 'blue bus near station'}],
   'applied_filters': {'tags': ['traffic'],


## Filter case: `legacy_filters_map`

This cell mirrors the `legacy_filters_map` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-001', 'vid-002']`

**Payload overlay:**

```json
{
  "filters": {
    "bucket_name": {
      "op": "in",
      "value": [
        "cam-a",
        "cam-b"
      ]
    },
    "frame_number": {
      "op": "between",
      "value": [
        10,
        20
      ]
    }
  }
}
```


In [30]:
run_filter_case("legacy_filters_map")


Running filter case: legacy_filters_map
Expected IDs: ['vid-001', 'vid-002']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "filters -> where(all=[legacy predicates])",
      "bucket_name:in",
      "frame_number:between"
    ],
    "filters": {
      "bucket_name": {
        "op": "in",
        "value": [
          "cam-a",
          "cam-b"
        ]
      },
      "frame_number": {
        "op": "between",
        "value": [
          10,
          20
        ]
      }
    },
    "normalized_where": {
      "all": [
        {
          "all": null,
          "any": null,
          "field": "bucket_name",
          "not": null,
          "op": "in",
          "value": [
            "cam-a",
            "cam-b"
          ]
        },
        {
          "all": null,
          "any": null,
          "field": "frame_number",
          "not": null,
          "op": "between",
          "value": [


{'request_id': '65139708-1e40-48e1-866d-b2e787332ea1',
 'results': [{'query_id': 'legacy_filters_map',
   'query': 'traffic light malfunctio',
   'count': 2,
   'items': [{'score': 0.4819686711,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north',
      'created_at': '2026-03-10T10:00:00+00:00',
      'description': 'red car at intersection',
      'frame_number': 10,
      'optional_note': 'doc has note',
      'prefix': 'alpha-route',
      'tags': 'traffic,vehicle,red',
      'video_id': 'vid-001'},
     'page_content': 'red car at intersection'},
    {'score': 0.3594945371,
     'metadata': {'bucket_name': 'cam-b',
      'camera_zone': 'south',
      'created_at': '2026-03-15T12:00:00+00:00',
      'description': 'blue bus near station',
      'frame_number': 20,
      'prefix': 'beta-route',
      'tags': 'traffic,vehicle,bus',
      'video_id': 'vid-002'},
     'page_content': 'blue bus near station'}],
   'applied_filters': {'tags': None,
    'time_filter': No

## Filter case: `logical_not_toplevel`

This cell mirrors the `logical_not_toplevel` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-002', 'vid-003', 'vid-004', 'vid-005', 'vid-006']`

**Payload overlay:**

```json
{
  "where": {
    "not": {
      "field": "camera_zone",
      "op": "eq",
      "value": "north"
    }
  }
}
```


In [31]:
run_filter_case("logical_not_toplevel")


Running filter case: logical_not_toplevel
Expected IDs: ['vid-002', 'vid-003', 'vid-004', 'vid-005', 'vid-006']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "not(...)"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": null,
      "not": {
        "all": null,
        "any": null,
        "field": "camera_zone",
        "not": null,
        "op": "eq",
        "value": "north"
      },
      "op": null,
      "value": null
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Logical clauses 'any'/'not' are evaluated in fallback path and not pushed down to backend."
    ]
  },
  "count": 5,
  "matched_ids": [
    "vid-005",
    "vid-004",
    "vid-003",
    "vid-006",
    "vid-002"
  ],
  "query_id": "logical_not_toplevel"
}


{'request_id': 'a7e8af3b-ecc7-4fd8-9700-e0589e031183',
 'results': [{'query_id': 'logical_not_toplevel',
   'query': 'traffic light malfunctio',
   'count': 5,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.6137043238,
     'metadata': {'bucket_name': 'cam-d',
      'camera_zone': 'west',
      'created_at': '2026-03-25T14:15:00+00:00',
      'description': 'delivery truck unloading',
      'frame_number': 40,
      'prefix': 'gamma-log',
      'tags': 'logistics,vehicle',
      'video_id': 'vid-004'},
     'page_content': 'delivery truck unloading'},
    {'score': 0.60422194,
     'metadata': {'bucke

## Filter case: `no_match`

This cell mirrors the `no_match` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `[]`

**Payload overlay:**

```json
{
  "where": {
    "field": "video_id",
    "op": "eq",
    "value": "vid-999"
  }
}
```


In [32]:
run_filter_case("no_match")


Running filter case: no_match
Expected IDs: []
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": null,
    "dropped_or_rewritten_clauses": [
      "video_id:eq"
    ],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "video_id",
      "not": null,
      "op": "eq",
      "value": "vid-999"
    },
    "tags": null,
    "time_filter": null,
    "warnings": [
      "Predicate video_id:eq is not backend-pushdown compatible and will be evaluated in fallback."
    ]
  },
  "count": 0,
  "matched_ids": [],
  "query_id": "no_match"
}


{'request_id': '88070f9e-c583-418d-86c0-9a1487f982d5',
 'results': [{'query_id': 'no_match',
   'query': 'traffic light malfunctio',
   'count': 0,
   'items': [],
   'applied_filters': {'tags': None,
    'time_filter': None,
    'filters': None,
    'normalized_where': {'field': 'video_id',
     'op': 'eq',
     'value': 'vid-999',
     'all': None,
     'any': None,
     'not': None},
    'warnings': ['Predicate video_id:eq is not backend-pushdown compatible and will be evaluated in fallback.'],
    'compiled_backend_filter': None,
    'dropped_or_rewritten_clauses': ['video_id:eq']}}],
 'errors': []}

## Filter case: `op_time_between_where`

This cell mirrors the `op_time_between_where` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-003', 'vid-004', 'vid-005']`

**Payload overlay:**

```json
{
  "where": {
    "field": "created_at",
    "op": "between",
    "value": [
      "2026-03-20T00:00:00+00:00",
      "2026-04-01T23:59:59+00:00"
    ]
  }
}
```


In [33]:
run_filter_case("op_time_between_where")


Running filter case: op_time_between_where
Expected IDs: ['vid-003', 'vid-004', 'vid-005']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": {
      "created_at": [
        ">=",
        "2026-03-20T00:00:00+00:00",
        "<=",
        "2026-04-01T23:59:59+00:00"
      ]
    },
    "dropped_or_rewritten_clauses": [],
    "filters": null,
    "normalized_where": {
      "all": null,
      "any": null,
      "field": "created_at",
      "not": null,
      "op": "between",
      "value": [
        "2026-03-20T00:00:00+00:00",
        "2026-04-01T23:59:59+00:00"
      ]
    },
    "tags": null,
    "time_filter": null,
    "warnings": null
  },
  "count": 3,
  "matched_ids": [
    "vid-005",
    "vid-004",
    "vid-003"
  ],
  "query_id": "op_time_between_where"
}


{'request_id': '9f25e8d3-fab7-4dac-862c-09938de7112b',
 'results': [{'query_id': 'op_time_between_where',
   'query': 'traffic light malfunctio',
   'count': 3,
   'items': [{'score': 0.9354854822,
     'metadata': {'bucket_name': 'cam-a',
      'camera_zone': 'north-east',
      'created_at': '2026-04-01T09:45:00+00:00',
      'description': 'traffic light malfunction',
      'frame_number': 50,
      'optional_note': 'signal issue',
      'prefix': 'alpha-signal',
      'tags': 'traffic,signal',
      'video_id': 'vid-005'},
     'page_content': 'traffic light malfunction'},
    {'score': 0.6137043238,
     'metadata': {'bucket_name': 'cam-d',
      'camera_zone': 'west',
      'created_at': '2026-03-25T14:15:00+00:00',
      'description': 'delivery truck unloading',
      'frame_number': 40,
      'prefix': 'gamma-log',
      'tags': 'logistics,vehicle',
      'video_id': 'vid-004'},
     'page_content': 'delivery truck unloading'},
    {'score': 0.60422194,
     'metadata': {'buck

## Filter case: `nested_all`

This cell mirrors the `nested_all` case from `FILTER_CASES` and performs one `/query` call.

**Expected IDs:** `['vid-003']`

**Payload overlay:**

```json
{
  "where": {
    "all": [
      {
        "all": [
          {
            "field": "frame_number",
            "op": "gte",
            "value": 10
          },
          {
            "field": "frame_number",
            "op": "lte",
            "value": 30
          }
        ]
      },
      {
        "field": "bucket_name",
        "op": "eq",
        "value": "cam-c"
      }
    ]
  }
}
```


In [34]:
run_filter_case("nested_all")


Running filter case: nested_all
Expected IDs: ['vid-003']
POST /query -> 200
Result summary
{
  "applied_filters": {
    "compiled_backend_filter": {
      "frame_number": [
        ">=",
        10
      ]
    },
    "dropped_or_rewritten_clauses": [
      "frame_number:lte",
      "bucket_name:eq"
    ],
    "filters": null,
    "normalized_where": {
      "all": [
        {
          "all": [
            {
              "all": null,
              "any": null,
              "field": "frame_number",
              "not": null,
              "op": "gte",
              "value": 10
            },
            {
              "all": null,
              "any": null,
              "field": "frame_number",
              "not": null,
              "op": "lte",
              "value": 30
            }
          ],
          "any": null,
          "field": null,
          "not": null,
          "op": null,
          "value": null
        },
        {
          "all": null,
          "any": null,
 

{'request_id': 'a902336c-2af3-49b9-acdc-3ede8a7dc4f3',
 'results': [{'query_id': 'nested_all',
   'query': 'traffic light malfunctio',
   'count': 1,
   'items': [{'score': 0.60422194,
     'metadata': {'bucket_name': 'cam-c',
      'camera_zone': 'east',
      'created_at': '2026-03-20T08:30:00+00:00',
      'description': 'person with bicycle crossing',
      'frame_number': 30,
      'optional_note': 'pedestrian event',
      'prefix': 'alpha-bike',
      'tags': 'pedestrian,bicycle',
      'video_id': 'vid-003'},
     'page_content': 'person with bicycle crossing'}],
   'applied_filters': {'tags': None,
    'time_filter': None,
    'filters': None,
    'normalized_where': {'field': None,
     'op': None,
     'value': None,
     'all': [{'field': None,
       'op': None,
       'value': None,
       'all': [{'field': 'frame_number',
         'op': 'gte',
         'value': 10,
         'all': None,
         'any': None,
         'not': None},
        {'field': 'frame_number',
      

## Inspect compose services before cleanup

This optional checkpoint is handy when you want to confirm container names and status before tearing the stack down.


In [35]:
compose_ps()


$ docker compose -f docker/compose.yaml -f docker/compose.vdms.yaml ps
NAME                           IMAGE                                       COMMAND                  SERVICE                        CREATED          STATUS                             PORTS
docker-vdms-vector-db-1        intellabs/vdms:v2.12.0                      "/start.sh"              vdms-vector-db                 18 seconds ago   Up 17 seconds (health: starting)   0.0.0.0:5511->55555/tcp, [::]:5511->55555/tcp
docker-vector-retriever-1      vector-retriever-vdms:latest                "/app/scripts/entryp…"   vector-retriever               18 seconds ago   Up 6 seconds                       0.0.0.0:6101->8000/tcp, [::]:6101->8000/tcp
multimodal-embedding-serving   intel/multimodal-embedding-serving:latest   "gunicorn -b 0.0.0.0…"   multimodal-embedding-serving   18 seconds ago   Up 17 seconds (healthy)            0.0.0.0:9711->8000/tcp, [::]:9711->8000/tcp


time="2026-05-25T11:26:41+05:30" level=warning msg="The \"REGISTRY\" variable is not set. Defaulting to a blank string."


CompletedProcess(args=['docker', 'compose', '-f', 'docker/compose.yaml', '-f', 'docker/compose.vdms.yaml', 'ps'], returncode=0, stdout='NAME                           IMAGE                                       COMMAND                  SERVICE                        CREATED          STATUS                             PORTS\ndocker-vdms-vector-db-1        intellabs/vdms:v2.12.0                      "/start.sh"              vdms-vector-db                 18 seconds ago   Up 17 seconds (health: starting)   0.0.0.0:5511->55555/tcp, [::]:5511->55555/tcp\ndocker-vector-retriever-1      vector-retriever-vdms:latest                "/app/scripts/entryp…"   vector-retriever               18 seconds ago   Up 6 seconds                       0.0.0.0:6101->8000/tcp, [::]:6101->8000/tcp\nmultimodal-embedding-serving   intel/multimodal-embedding-serving:latest   "gunicorn -b 0.0.0.0…"   multimodal-embedding-serving   18 seconds ago   Up 17 seconds (healthy)            0.0.0.0:9711->8000/tcp, [::]:9711

## Teardown and cleanup

Run this cell when you are finished exploring. It stops the compose stack, removes volumes,
and clears the notebook's in-memory runtime state.


In [7]:
teardown_stack(remove_volumes=True)


$ docker compose -f docker/compose.yaml -f docker/compose.vdms.yaml down -v --remove-orphans


Notebook state cleared.


time="2026-05-26T14:14:55+05:30" level=warning msg="The \"REGISTRY\" variable is not set. Defaulting to a blank string."
 Container docker-vector-retriever-1 Stopping 
 Container docker-vector-retriever-1 Stopped 
 Container docker-vector-retriever-1 Removing 
 Container docker-vector-retriever-1 Removed 
 Container docker-vdms-vector-db-1 Stopping 
 Container multimodal-embedding-serving Stopping 
 Container multimodal-embedding-serving Stopped 
 Container multimodal-embedding-serving Removing 
 Container multimodal-embedding-serving Removed 
 Container docker-vdms-vector-db-1 Stopped 
 Container docker-vdms-vector-db-1 Removing 
 Container docker-vdms-vector-db-1 Removed 
 Volume docker_ov-models Removing 
 Volume docker_vdms-db Removing 
 Volume docker_data-prep Removing 
 Network docker_vr_network Removing 
 Volume docker_ov-models Removed 
 Volume docker_vdms-db Removed 
 Volume docker_data-prep Removed 
 Network docker_vr_network Removed
